# Part 2.3 — Creative AI: QuickDraw Pizza Sketch Generation with DCGAN

This notebook trains a **DCGAN** to generate synthetic **QuickDraw pizza sketches**.

## Notebook goals
1. Load the `full_simplified_pizza.ndjson` file.
2. Rasterise vector drawings into 28×28 sketch images.
3. Explore and visualise real pizza sketches.
4. Train a DCGAN to generate fake pizza sketches.
5. Monitor generator and discriminator loss curves.
6. Compare real and generated sketches visually.
7. Save report-ready figures and CSV summaries.

In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

BASE_OUTPUT_DIR = Path("outputs") / "quickdraw_pizza"
CSV_DIR = BASE_OUTPUT_DIR / "csv"
PLOTS_DIR = BASE_OUTPUT_DIR / "plots"

for folder in [BASE_OUTPUT_DIR, CSV_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("full_simplified_pizza.ndjson")

COLOR_REAL = "#7A1F1F"
COLOR_FAKE = "#C97B63"
COLOR_ALT = "#6C8A3B"
COLOR_GRID = "#D9D2C3"
COLOR_TEXT = "#2F2A24"

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.facecolor"] = "#FAF7F2"
plt.rcParams["figure.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#B9AD99"
plt.rcParams["grid.color"] = COLOR_GRID
plt.rcParams["axes.labelcolor"] = COLOR_TEXT
plt.rcParams["xtick.color"] = COLOR_TEXT
plt.rcParams["ytick.color"] = COLOR_TEXT
plt.rcParams["text.color"] = COLOR_TEXT
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = False

def save_plot(filename):
    filepath = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {filepath}")

In [ ]:
# ============================================================
# OPTIONAL GOOGLE DRIVE SETUP
# ============================================================
"""
Use this cell only if running from Google Drive.

If your file is in Drive, uncomment the mount lines and update DATA_DIR.
If you uploaded the .ndjson file directly to Colab, leave this cell unchanged.
"""

# from google.colab import drive
# drive.mount('/content/drive')

# DATA_DIR = Path("/content/drive/MyDrive/QuickDraw_Pizza")
# DATA_PATH = DATA_DIR / "full_simplified_pizza.ndjson"

# OUTPUT_DIR = Path("/content/drive/MyDrive/QuickDraw_Pizza_outputs")
# CSV_DIR = OUTPUT_DIR / "csv"
# PLOTS_DIR = OUTPUT_DIR / "plots"
# for folder in [OUTPUT_DIR, CSV_DIR, PLOTS_DIR]:
#     folder.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

In [ ]:
# ============================================================
# LOAD QUICKDRAW NDJSON FILE
# ============================================================

MAX_DRAWINGS = 60000

records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= MAX_DRAWINGS:
            break
        records.append(json.loads(line))

print(f"Loaded records: {len(records)}")
print("Example keys:", records[0].keys())

metadata_summary = pd.DataFrame({
    "metric": ["loaded_records", "max_drawings_setting"],
    "value": [len(records), MAX_DRAWINGS],
})
metadata_summary.to_csv(CSV_DIR / "01_loaded_dataset_summary.csv", index=False)
display(metadata_summary)

In [ ]:
# ============================================================
# EXPLORE RECOGNITION METADATA
# ============================================================

meta_df = pd.DataFrame({
    "recognized": [record.get("recognized", None) for record in records],
    "word": [record.get("word", None) for record in records],
    "countrycode": [record.get("countrycode", None) for record in records],
})

recognition_counts = meta_df["recognized"].value_counts(dropna=False).reset_index()
recognition_counts.columns = ["recognized", "count"]

country_counts = meta_df["countrycode"].value_counts().head(10).reset_index()
country_counts.columns = ["countrycode", "count"]

recognition_counts.to_csv(CSV_DIR / "02_recognition_counts.csv", index=False)
country_counts.to_csv(CSV_DIR / "03_top_country_counts.csv", index=False)

display(recognition_counts)
display(country_counts)

In [ ]:
# ============================================================
# FILTER TO RECOGNISED PIZZA DRAWINGS
# ============================================================

recognized_records = [record for record in records if record.get("recognized", True) is True]

if len(recognized_records) >= 1000:
    selected_records = recognized_records
    filter_used = "recognized_true"
else:
    selected_records = records
    filter_used = "all_loaded_records"

filter_summary = pd.DataFrame({
    "metric": ["filter_used", "selected_records"],
    "value": [filter_used, len(selected_records)],
})
filter_summary.to_csv(CSV_DIR / "04_filter_summary.csv", index=False)
display(filter_summary)

In [ ]:
# ============================================================
# RASTERISE QUICKDRAW STROKES TO 28x28 IMAGES
# ============================================================

IMAGE_SIZE = 28

def rasterise_drawing(drawing, image_size=28, line_width=6):
    canvas = Image.new("L", (256, 256), color=0)
    draw = ImageDraw.Draw(canvas)

    for stroke in drawing:
        x_points, y_points = stroke
        points = list(zip(x_points, y_points))
        if len(points) > 1:
            draw.line(points, fill=255, width=line_width, joint="curve")

    canvas = canvas.resize((image_size, image_size), Image.Resampling.LANCZOS)
    arr = np.array(canvas).astype(np.float32) / 255.0
    return arr

rasterised_images = np.stack([
    rasterise_drawing(record["drawing"], image_size=IMAGE_SIZE)
    for record in selected_records
])

print("Rasterised image shape:", rasterised_images.shape)
print("Pixel min/max:", rasterised_images.min(), rasterised_images.max())

image_summary = pd.DataFrame({
    "metric": ["n_images", "image_size", "pixel_min", "pixel_max", "pixel_mean", "pixel_std"],
    "value": [
        int(rasterised_images.shape[0]),
        int(IMAGE_SIZE),
        float(rasterised_images.min()),
        float(rasterised_images.max()),
        float(rasterised_images.mean()),
        float(rasterised_images.std()),
    ],
})
image_summary.to_csv(CSV_DIR / "05_rasterised_image_summary.csv", index=False)
display(image_summary)

In [ ]:
# ============================================================
# VISUALISE REAL PIZZA SKETCHES
# ============================================================

def show_image_grid(images, filename, title, n_images=64):
    indices = np.random.choice(len(images), n_images, replace=False)
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    axes = axes.flatten()

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx], cmap="gray")
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    save_plot(filename)
    plt.show()

show_image_grid(
    rasterised_images,
    filename="01_real_pizza_sketch_grid.png",
    title="Real QuickDraw Pizza Sketches",
    n_images=64,
)

In [ ]:
# ============================================================
# CUSTOM DATASET AND DATALOADER
# ============================================================

class QuickDrawPizzaDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        image = (image * 2.0) - 1.0
        return image, 0

BATCH_SIZE = 256

pizza_dataset = QuickDrawPizzaDataset(rasterised_images)
pizza_loader = DataLoader(pizza_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Training images: {len(pizza_dataset)}")
print(f"Batches per epoch: {len(pizza_loader)}")

In [ ]:
# ============================================================
# DCGAN MODEL DEFINITIONS
# ============================================================

LATENT_DIM = 100
IMAGE_CHANNELS = 1
FEATURE_MAPS_GEN = 64
FEATURE_MAPS_DISC = 64

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=IMAGE_CHANNELS, feature_maps=FEATURE_MAPS_GEN):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feature_maps * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 2, feature_maps, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps, channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.model(z)
        return x[:, :, :28, :28]


class Discriminator(nn.Module):
    def __init__(self, channels=IMAGE_CHANNELS, feature_maps=FEATURE_MAPS_DISC):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, feature_maps, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 2, feature_maps * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 4, 1, 3, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.model(x)
        return out.view(-1, 1)

generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)

print(generator)
print()
print(discriminator)

In [ ]:
# ============================================================
# WEIGHT INITIALISATION
# ============================================================

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

generator.apply(weights_init)
discriminator.apply(weights_init)

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 50
LEARNING_RATE = 0.0002
BETA1 = 0.5

criterion = nn.BCELoss()
optimizer_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
optimizer_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)

print(f"NUM_EPOCHS = {NUM_EPOCHS}")
print(f"LEARNING_RATE = {LEARNING_RATE}")

In [ ]:
# ============================================================
# VISUALISATION HELPERS
# ============================================================

def denormalise_images(x):
    return (x + 1.0) / 2.0

def save_generated_grid(generator, noise, epoch, filename):
    generator.eval()
    with torch.no_grad():
        fake = generator(noise).detach().cpu()
        fake = denormalise_images(fake)
        grid = make_grid(fake, nrow=8, padding=2)

    plt.figure(figsize=(8, 8))
    plt.imshow(grid.squeeze(0).numpy(), cmap="gray")
    plt.title(f"Generated Pizza Sketches — Epoch {epoch}")
    plt.axis("off")
    save_plot(filename)
    plt.show()
    generator.train()

In [ ]:
# ============================================================
# TRAIN THE QUICKDRAW PIZZA DCGAN
# ============================================================

history = []
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    batches = 0

    for real_images, _ in pizza_loader:
        real_images = real_images.to(DEVICE)
        batch_size = real_images.size(0)

        real_labels = torch.ones((batch_size, 1), device=DEVICE)
        fake_labels = torch.zeros((batch_size, 1), device=DEVICE)

        discriminator.zero_grad()

        output_real = discriminator(real_images)
        loss_d_real = criterion(output_real, real_labels)

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
        fake_images = generator(noise)

        output_fake = discriminator(fake_images.detach())
        loss_d_fake = criterion(output_fake, fake_labels)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        optimizer_d.step()

        generator.zero_grad()
        output_fake_for_g = discriminator(fake_images)
        loss_g = criterion(output_fake_for_g, real_labels)
        loss_g.backward()
        optimizer_g.step()

        epoch_d_loss += loss_d.item()
        epoch_g_loss += loss_g.item()
        batches += 1

    avg_d_loss = epoch_d_loss / batches
    avg_g_loss = epoch_g_loss / batches

    history.append({
        "epoch": epoch,
        "loss_discriminator": avg_d_loss,
        "loss_generator": avg_g_loss,
    })

    if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss D: {avg_d_loss:.4f} | Loss G: {avg_g_loss:.4f}")
        save_generated_grid(generator, fixed_noise, epoch, f"02_generated_pizza_epoch_{epoch:03d}.png")

training_time_seconds = time.time() - start_time
history_df = pd.DataFrame(history)
history_df.to_csv(CSV_DIR / "06_training_loss_history.csv", index=False)

print(f"Training completed in {training_time_seconds:.2f} seconds.")

In [ ]:
# ============================================================
# PLOT TRAINING LOSSES
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["loss_discriminator"], label="Discriminator Loss", color=COLOR_REAL, linewidth=2)
plt.plot(history_df["epoch"], history_df["loss_generator"], label="Generator Loss", color=COLOR_ALT, linewidth=2)
plt.title("QuickDraw Pizza DCGAN Training Losses")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
save_plot("03_quickdraw_pizza_loss_curves.png")
plt.show()

In [ ]:
# ============================================================
# FINAL REAL VS GENERATED COMPARISON
# ============================================================

def show_real_vs_generated(images, generator, n_images=64):
    real_idx = np.random.choice(len(images), n_images, replace=False)
    real_batch = torch.tensor(images[real_idx], dtype=torch.float32).unsqueeze(1)

    noise = torch.randn(n_images, LATENT_DIM, 1, 1, device=DEVICE)
    generator.eval()
    with torch.no_grad():
        fake_batch = generator(noise).cpu()
        fake_batch = denormalise_images(fake_batch)

    real_grid = make_grid(real_batch, nrow=8, padding=2)
    fake_grid = make_grid(fake_batch, nrow=8, padding=2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(real_grid.squeeze(0).numpy(), cmap="gray")
    axes[0].set_title("Real QuickDraw Pizza Sketches")
    axes[0].axis("off")

    axes[1].imshow(fake_grid.squeeze(0).numpy(), cmap="gray")
    axes[1].set_title("Generated Pizza Sketches")
    axes[1].axis("off")

    save_plot("04_real_vs_generated_pizza_sketches.png")
    plt.show()

show_real_vs_generated(rasterised_images, generator, n_images=64)

In [ ]:
# ============================================================
# SIMPLE PIXEL-STATISTIC COMPARISON
# ============================================================

n_eval = min(5000, len(rasterised_images))
real_idx = np.random.choice(len(rasterised_images), n_eval, replace=False)

real_eval = rasterised_images[real_idx]

noise = torch.randn(n_eval, LATENT_DIM, 1, 1, device=DEVICE)
generator.eval()
with torch.no_grad():
    fake_eval = generator(noise).cpu()
    fake_eval = denormalise_images(fake_eval).squeeze(1).numpy()

real_pixel_mean = float(real_eval.mean())
fake_pixel_mean = float(fake_eval.mean())

real_pixel_std = float(real_eval.std())
fake_pixel_std = float(fake_eval.std())

real_ink_ratio = float((real_eval > 0.15).mean())
fake_ink_ratio = float((fake_eval > 0.15).mean())

pixel_metrics = pd.DataFrame([{
    "real_pixel_mean": real_pixel_mean,
    "fake_pixel_mean": fake_pixel_mean,
    "real_pixel_std": real_pixel_std,
    "fake_pixel_std": fake_pixel_std,
    "real_ink_ratio": real_ink_ratio,
    "fake_ink_ratio": fake_ink_ratio,
    "pixel_mean_abs_diff": abs(real_pixel_mean - fake_pixel_mean),
    "ink_ratio_abs_diff": abs(real_ink_ratio - fake_ink_ratio),
}])

pixel_metrics.to_csv(CSV_DIR / "07_pixel_statistic_comparison.csv", index=False)
display(pixel_metrics)

In [ ]:
# ============================================================
# REPORT-READY SUMMARY TABLE
# ============================================================

report_summary = pd.DataFrame({
    "item": [
        "loaded_records",
        "selected_records",
        "image_size",
        "channels",
        "batch_size",
        "latent_dim",
        "epochs",
        "training_time_seconds",
        "real_ink_ratio",
        "fake_ink_ratio",
    ],
    "value": [
        int(len(records)),
        int(len(selected_records)),
        int(IMAGE_SIZE),
        int(IMAGE_CHANNELS),
        int(BATCH_SIZE),
        int(LATENT_DIM),
        int(NUM_EPOCHS),
        round(training_time_seconds, 2),
        round(real_ink_ratio, 4),
        round(fake_ink_ratio, 4),
    ],
})

report_summary.to_csv(CSV_DIR / "08_report_ready_summary_table.csv", index=False)
display(report_summary)

## Interpretation guide for the report

### Why DCGAN was used
QuickDraw pizza sketches are image-like outputs after rasterisation. A convolutional generator and discriminator are appropriate because sketch structure depends on local spatial patterns.

### Dataset processing
The raw QuickDraw data is vector-based. It was rasterised into 28×28 grayscale images so it could be used with a DCGAN.

### Evaluation
Use generated sketch grids, real-vs-generated comparison, training loss curves, and pixel/ink statistics. For sparse sketches, ImageNet-based FID may not reflect human-perceived sketch quality very well.

### Limitations
Sketches are sparse and abstract, so the model may learn stroke density before learning semantic pizza shape. Some drawings vary strongly in style and complexity.

In [ ]:
# ============================================================
# OPTIONAL LIST OF SAVED OUTPUT FILES
# ============================================================

print("Saved CSV files:")
for path in sorted(CSV_DIR.glob("*.csv")):
    print("-", path)

print("\nSaved plot files:")
for path in sorted(PLOTS_DIR.glob("*.png")):
    print("-", path)